# Milestone - Unit 13: race day

Unit 13 assembled the book's race-day agent for `monza-like`, the stand-in for the real race's one known track.
`agents/unit-13.py` is `agents/unit-11.py` with exactly six module constants changed, and nothing else:

- `LINE_CONTROL`: `monza-like`'s racing line (Unit 9's solver, 285 control values);
- `MU_G`: `monza-like`'s grip, Unit 8's six probes (section 0 a lower bound, which costs nothing on this line);
- `MU_FRACTION = 0.64`, `A_DEC = 4.5`, `A_ACC = 5.0`: Unit 12's crash-aware re-tune on the known track found the edge (0.66, 5.0, 5.0), and Unit 11's margin rule put one grid step inside it on **every** knob at its edge (`MU_FRACTION` and `A_DEC`);
- `MPC_MIN_HORIZON = 40`, equal to `MPC_HORIZON`: the tick budget only counts a slow call, so the race that ships is the race the gates ran.

This milestone asserts design 013's gate for the unit (plan `roar-013`), on `monza-like`, 3 laps:

- **U13.0** the file: the six constants against `unit-11.py`, read from the two files; the rest of the file the same line for line; the line equal to the one Unit 9's solver gives here (to 1e-6 m); the grip Unit 8's;
- **U13.1** `unit-13.py` against `agents/unit-07.py` (Part 1's agent) on the nominal car, race seeds 0-4: both clean in all five, `noise_aware_better(unit-13's times, unit-07's times)`; medians 468.55 and 515.10 s;
- **U13.6** `check_submission(agents/unit-13.py, track="monza-like")` passes, with `step()`'s 99th percentile at most 5 ms: the timing on, as shipped, the one timed run in this unit.

Every race uses racekit's carla-like car with its noise, `carla_like(noise=True)`, cut at 900 s, and `unit-13.py` races with `MPC_BUDGET_MS = None` (no wall clock inside a pinned race; with `MPC_MIN_HORIZON = MPC_HORIZON` the timing changes only a counter, so this is the shipped race); `unit-07.py` has no clock and races as shipped.
Tolerances (the plan's Gates): times within one tick (±0.05 s) of the pinned value.
A race is **clean** when it finished, with 0 collisions, and its max |cross-track error from its own path| over the driving rows is at most 2 m (`unit-13.py`'s path is its line; `unit-07.py`'s is the centreline).
The routed premise (`tests/test_racekit_unit13_premise.py`) races the rest: the 24 test cars k = 72-95 (U13.2), the one-knob margin's crash (U13.3), pursuit's (U13.4) and section 0's grip (U13.5).
Every figure is a model number on racekit's carla-like car, not CARLA's.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
The line solver is Unit 9's; the file reader is Unit 12's (each constant's value as text, without its comment).

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("..") / "agents"
UNIT_07, UNIT_11, UNIT_13 = AGENTS / "unit-07.py", AGENTS / "unit-11.py", AGENTS / "unit-13.py"
TRACK = "monza-like"
MONZA_MU_G = (5.38, 8.96, 9.12, 9.14, 9.70, 7.97)  # m/s^2: Unit 8's six grip probes on monza-like
SEEDS = (0, 1, 2, 3, 4)
TICK = 0.05 + 1e-9  # s: the plan's tolerance, one tick
CLEAN_ERROR = 2.0  # m
CAP = 900.0  # s
STEP_P99_BUDGET_MS = 5.0
NO_TIMING = {"MPC_BUDGET_MS": None}
workdir = Path(mkdtemp(prefix="roar-milestone-13-"))

# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x

NAME_CHARACTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"


def constant_name(line):
    # The UPPER_CASE name of a module-constant line ("NAME = value  # comment"), or None (Unit 12's rule).
    cuts = [i for i in range(len(line) - 2) if line[i:i + 3] == " = "]
    if cuts and line[0] in NAME_CHARACTERS[:26] and all(c in NAME_CHARACTERS for c in line[:cuts[0]]):
        return line[:cuts[0]]
    return None


def module_constants(path):
    # name -> the value's text (after " = ", without a "  # ..." comment), for every module-constant line.
    # A value written over several lines (LINE_CONTROL) is only its first line here: "(".
    constants = {}
    with open(path) as handle:
        for line in handle:
            name = constant_name(line)
            if name is not None:
                text = line[len(name) + 3:-1]
                comment = [i for i in range(len(text) - 2) if text[i:i + 3] == "  #"]
                constants[name] = text[:comment[0]] if comment else text
    return constants


def file_parts(path):
    # (the comment lines above the docstring, the rest of the file without its constant lines and without
    # the line's values): the header may change, the rest must not.
    with open(path) as handle:
        lines = [line for line in handle]
    start = 0
    for i, line in enumerate(lines):
        if line[:3] == '"""':
            start = i
            break
    body, in_line = [], False
    for line in lines[start:]:
        if line[:15] == "LINE_CONTROL = ":
            in_line = True
        elif in_line:
            in_line = line != ")\n"
        elif constant_name(line) is None:
            body.append(line)
    return lines[:start], body


track = load_track(TRACK)
with shim_installed():
    Unit11, name11 = load_submission(UNIT_11)
    Unit13, name13 = load_submission(UNIT_13)
eleven, thirteen = sys.modules[name11], sys.modules[name13]
LINE = np.array(Unit13(track.waypoints, None).offsets)  # m: unit-13.py's path, as offsets from the centreline
CENTRELINE = np.zeros(len(track))  # unit-07.py's path


def line_error(result, offsets):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result, offsets):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def five_races(path):
    # Three laps of monza-like, race seeds 0-4, the nominal carla-like car with its noise.
    return repeat_races(path, TRACK, n=5, seeds=SEEDS, car_factory=lambda: carla_like(noise=True), laps=3,
                        max_seconds=CAP)


def report(name, runs, offsets):
    for seed, result in zip(SEEDS, runs.results):
        print(f"{name} seed {seed}: {result.total_time:.2f} s, clean {clean(result, offsets)}, "
              f"{result.collisions} collisions, {line_error(result, offsets):.2f} m from its path")
    print(f"{name}: median {runs.median:.2f} s, spread {runs.spread:.2f} s")

## 2. U13.0: the file

Read both files: every constant whose text changed, the line (written over many lines, so compared as numbers), the rest of each file line for line, and the line solved again here (about 2 s).

In [ ]:
eleven_text, thirteen_text = module_constants(UNIT_11), module_constants(UNIT_13)
changed = [name for name in eleven_text if eleven_text[name] != thirteen_text[name]]
if len(eleven.LINE_CONTROL) != len(thirteen.LINE_CONTROL) or eleven.LINE_CONTROL != thirteen.LINE_CONTROL:
    changed.append("LINE_CONTROL")
for name in changed[:-1]:
    print(f"{name:16s} {eleven_text[name]:36s} -> {thirteen_text[name]}")
print(f"{'LINE_CONTROL':16s} {len(eleven.LINE_CONTROL)} values (test-circuit){'':15s} -> {len(thirteen.LINE_CONTROL)} values")
assert set(eleven_text) == set(thirteen_text) and len(thirteen_text) == 48
assert sorted(changed) == sorted(["LINE_CONTROL", "MU_G", "MU_FRACTION", "A_DEC", "A_ACC", "MPC_MIN_HORIZON"])  # U13.0

header13, body13 = file_parts(UNIT_13)
header11, body11 = file_parts(UNIT_11)
print(f"unit-13.py: {len(header13)} header comment lines, then {len(body13)} lines the same as unit-11.py's: "
      f"{body13 == body11}")
assert body13 == body11 and header11 == [] and all(line[0] == "#" for line in header13)  # U13.0

values = {"MU_FRACTION": 0.64, "A_DEC": 4.5, "A_ACC": 5.0, "MPC_MIN_HORIZON": 40, "MPC_HORIZON": 40,
          "MPC_BUDGET_MS": 5.0, "STEERING": "mpc"}
held = {"MU_FRACTION": thirteen.MU_FRACTION, "A_DEC": thirteen.A_DEC, "A_ACC": thirteen.A_ACC,
        "MPC_MIN_HORIZON": thirteen.MPC_MIN_HORIZON, "MPC_HORIZON": thirteen.MPC_HORIZON,
        "MPC_BUDGET_MS": thirteen.MPC_BUDGET_MS, "STEERING": thirteen.STEERING}
print("the shipped values:", held)
assert held == values and thirteen.MPC_MIN_HORIZON == thirteen.MPC_HORIZON  # U13.0
assert thirteen.MU_G == MONZA_MU_G  # U13.0: Unit 8's probes

control = solve_line(track)
difference = float(np.max(np.abs(control - np.array(thirteen.LINE_CONTROL))))
print(f"{TRACK}: {len(control)} control values solved here; the largest |solved - shipped| {difference:.1e} m")
assert len(thirteen.LINE_CONTROL) == len(control) == 285 and difference <= 1e-6  # U13.0

## 3. U13.1: `unit-13.py` against `unit-07.py`

Five races each, 3 laps, the nominal car (about 55 s for `unit-13.py`, 25 s for `unit-07.py`).

In [ ]:
no_timing = write_variant(UNIT_13, workdir / "u13-no-timing.py", {key: repr(value) for key, value in NO_TIMING.items()})
thirteen_runs = five_races(no_timing)
report("unit-13.py", thirteen_runs, LINE)
assert all(clean(result, LINE) for result in thirteen_runs.results)  # U13.1: 5/5 clean
assert abs(thirteen_runs.median - 468.55) <= TICK  # U13.1

In [ ]:
seven_runs = five_races(UNIT_07)
report("unit-07.py", seven_runs, CENTRELINE)
assert all(clean(result, CENTRELINE) for result in seven_runs.results)  # U13.1: 5/5 clean
assert abs(seven_runs.median - 515.10) <= TICK  # U13.1

gate = noise_aware_better(thirteen_runs.times, seven_runs.times)
print(f"unit-13.py is {seven_runs.median - thirteen_runs.median:.2f} s faster (median); the larger spread is "
      f"{max(thirteen_runs.spread, seven_runs.spread):.2f} s; Unit 1's rule: unit-13.py beats unit-07.py: {gate}")
print("seed  unit-13.py  unit-07.py")
for seed, a, b in zip(SEEDS, thirteen_runs.times, seven_runs.times):
    print(f"{seed:4d}  {a:10.2f}  {b:10.2f}")
assert gate  # U13.1: design 013's gate for Unit 13

## 4. U13.6: `check_submission` on the race track, with the timing on

The shipped file, exactly as shipped (`MPC_BUDGET_MS = 5.0`), on its own track: a Python 3.8 deny-list over its syntax, its imports (numpy, `roar_py_interface` and the Python 3.8 standard library only), the official shape (`RoarCompetitionSolution`, `async def initialize` and `step`), and one race on `monza-like` with `step()`'s 99th percentile at most 5 ms.
(On its default track, `test-circuit`, the file is refused before it races: its line was solved for `monza-like`.)
It is the one run in this unit with the clock inside the race, so its race time is not pinned and its p99 is printed, not pinned; **run it when nothing else is running on the machine** (about 10 s).

In [ ]:
result = check_submission(UNIT_13, track=TRACK)
print(result)
print(f"step() p99 {result.step_p99_ms:.3f} ms (the budget {STEP_P99_BUDGET_MS} ms)")
assert result.ok, result.findings  # U13.6
assert result.step_p99_ms <= STEP_P99_BUDGET_MS  # U13.6: the timing budget

## Unit 13, pinned

| `monza-like`, 3 laps, race seeds 0-4, `carla_like(noise=True)` | `unit-07.py` (Part 1) | `unit-13.py` (race day) |
|---|---|---|
| clean | 5/5 | 5/5 |
| median (spread) | 515.10 s (0.15) | **468.55 s** (0.10) |
| Unit 1's rule | | `unit-13.py` beats `unit-07.py`, by 46.55 s |
| `check_submission(track="monza-like")` | | PASS, p99 at most 5 ms (about 1.0 ms in the record) |

The routed premise races the rest of N/N: `unit-13.py` is clean on all 24 **test** cars k = 72-95 (U13.2: median 468.65 s, as the record raced them once with the shipped configuration); the validation cars k = 24-71 (24/24 and 24/24) are the plan's pinned record.
It also races why the file is what it is: the margin on `MU_FRACTION` alone crashes on held-out car 41 (U13.3), pursuit at the race-day setting crashes on car 34 (U13.4), and section 0's grip at 12.0 gives exactly the same race as its lower bound 5.38 (U13.5).
The race on CARLA is the student's, in the unit's lab; the author's timed CARLA check of this file is recorded in `roar/docs/qualification.md` as PENDING (author).